In [1]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
import sqlite3

con = sqlite3.connect("data/broadband.db")

subs_df = pd.read_csv("data/B28002 Presence and Types of Internet Subscriptions in Household/ACSDT5Y2024.B28002-Data.csv", header=1)
income_df = pd.read_csv("data/B28004 Household Income in the Last 12 Months (in 2024 Inflation-Adjusted Dollars) by Presence and Type of Internet Subscription in Household/ACSDT5Y2024.B28004-Data.csv", header=1)
age_df = pd.read_csv("data/B28005 Age by Presence of a Computer and Types of Internet Subscription in Household/ACSDT5Y2024.B28005-Data.csv", header=1)
labor_df = pd.read_csv("data/B28007 Labor Force Status by Presence of a Computer and Types of Internet Subscription in Household/ACSDT5Y2024.B28007-Data.csv", header=1)

In [2]:
subs_df["GEOID"] = subs_df["Geography"].str[9:]
subs_df = subs_df.drop(columns=["Geography", "Geographic Area Name"])
subs_df.insert(0,"GEOID", subs_df.pop("GEOID"))

status = {
    "With an Internet subscription": "sub",
    "Internet access without a subscription": "access_no_sub",
    "No Internet access": "no_access" 
}
sub_type = {
    "Dial-up with no other type of Internet subscription": "dialup_only",
    "Broadband of any type": "broadband_any",
    "Cellular data plan": "cell",
    "Cellular data plan with no other type of Internet subscription": "only",
    "Broadband such as cable, fiber optic or DSL": "wired",
    "Broadband such as cable, fiber optic or DSL with no other type of Internet subscription": "only",
    "Satellite Internet service": "sat",
    "Satellite Internet service with no other type of Internet subscription": "only",
    "Other service with no other type of Internet subscription": "other_only",
}

columns = subs_df.columns.tolist()

for i, label in enumerate(columns):
    if label == "GEOID":
        name = "geoid"
        subs_df = subs_df.rename(columns={subs_df.columns[i]: name})
        continue
    if "Unnamed" in label:
        subs_df = subs_df.drop(columns=subs_df.columns[i])
        continue 
    parts = label.split("!!")
    pieces = [status.get(p, sub_type.get(p)) for p in parts[2:]]
    name = "_".join(pieces) if pieces else "total"
    if parts[0] == "Margin of Error":
        name += "_moe"
    subs_df = subs_df.rename(columns={subs_df.columns[i]: name})

In [3]:
income_df["GEOID"] = income_df["Geography"].str[9:]
income_df = income_df.drop(columns=["Geography", "Geographic Area Name"])
income_df.insert(0,"GEOID", income_df.pop("GEOID"))

brackets = {
    "Less than $10,000:": "lt_10k",
    "$10,000 to $19,999:": "10_20k",
    "$20,000 to $34,999:": "20_35k",
    "$35,000 to $49,999:": "35_50k",
    "$50,000 to $74,999:": "50_75k",
    "$75,000 or more:": "75k_plus",
}
sub_type = {
    "With dial-up Internet subscription alone": "dialup",
    "With a broadband Internet subscription": "broadband",
    "Without an Internet subscription": "no_sub",
}

columns = income_df.columns.tolist()

for i, label in enumerate(columns):
    if label == "GEOID":
        name = "geoid"
        income_df = income_df.rename(columns={income_df.columns[i]: name})
        continue
    if "Unnamed" in label:
        income_df = income_df.drop(columns=income_df.columns[i])
        continue 
    parts = label.split("!!")
    pieces = [brackets.get(p, sub_type.get(p)) for p in parts[2:]]
    name = "_".join(pieces) if pieces else "total"
    if parts[0] == "Margin of Error":
        name += "_moe"
    income_df = income_df.rename(columns={income_df.columns[i]: name})

In [4]:
age_df["GEOID"] = age_df["Geography"].str[9:]
age_df = age_df.drop(columns=["Geography", "Geographic Area Name"])
age_df.insert(0,"GEOID", age_df.pop("GEOID"))

ages = {
    "Under 18 years:": "u18",
    "18 to 64 years:": "18_64",
    "65 years and over:": "65_plus", 
}
computer_sub = {
    "Has a computer:": "computer",
    "No computer": "no_computer",
    "With dial-up Internet subscription alone": "dialup",
    "With a broadband Internet subscription": "broadband",
    "Without an Internet subscription": "no_sub",
}

columns = age_df.columns.tolist()

for i, label in enumerate(columns):
    if label == "GEOID":
        name = "geoid"
        age_df = age_df.rename(columns={age_df.columns[i]: name})
        continue
    if "Unnamed" in label:
        age_df = age_df.drop(columns=age_df.columns[i])
        continue 
    parts = label.split("!!")
    pieces = [ages.get(p, computer_sub.get(p)) for p in parts[2:]]
    name = "_".join(pieces) if pieces else "total"
    if parts[0] == "Margin of Error":
        name += "_moe"
    age_df = age_df.rename(columns={age_df.columns[i]: name})

In [5]:
labor_df["GEOID"] = labor_df["Geography"].str[9:]
labor_df = labor_df.drop(columns=["Geography", "Geographic Area Name"])
labor_df.insert(0,"GEOID", labor_df.pop("GEOID"))

labor = {
    "In the civilian labor force:": "lf",
    "Employed:": "employed",
    "Unemployed:": "unemployed",
    "Not in labor force:": "not_lf", 
}
computer_sub = {
    "Has a computer:": "computer",
    "No computer": "no_computer",
    "With dial-up Internet subscription alone": "dialup",
    "With a broadband Internet subscription": "broadband",
    "Without an Internet subscription": "no_sub",
}

columns = labor_df.columns.tolist()

for i, label in enumerate(columns):
    if label == "GEOID":
        name = "geoid"
        labor_df = labor_df.rename(columns={labor_df.columns[i]: name})
        continue
    if "Unnamed" in label:
        labor_df = labor_df.drop(columns=labor_df.columns[i])
        continue 
    parts = label.split("!!")
    pieces = [labor.get(p, computer_sub.get(p)) for p in parts[2:]]
    name = "_".join(pieces) if pieces else "total"
    if parts[0] == "Margin of Error":
        name += "_moe"
    labor_df = labor_df.rename(columns={labor_df.columns[i]: name})

In [8]:
demographics = {
    "subs": subs_df,
    "income": income_df,
    "age": age_df,
    "labor": labor_df
}

state_demographics = {}

for name, dataframe in demographics.items():
    df = dataframe.copy()
    cols_to_edit = df.columns.drop("geoid")
    df[cols_to_edit] = df[cols_to_edit].apply(pd.to_numeric, errors='coerce').astype("Int64")
    df.loc[df["total"] == 0, cols_to_edit] = pd.NA

    df.iloc[1:].to_sql(f"acs_{name}", con, if_exists="replace", index=False)

    state = df.iloc[:1].copy()
    state = state.drop(columns="geoid")
    state = state.add_prefix(f"{name}_")
    state = state.reset_index(drop=True)

    state_demographics[f"{name}_state"] = state

In [7]:
acs_state = pd.concat(state_demographics.values(), axis=1)
acs_state.to_sql("acs_state", con, if_exists="replace", index=False)

1